# Re-derive the OBEREK dashboard from raw data and metadata

This notebook downloads the raw water-quality readings for OBEREK stations, reads the reference
values from the OBEREK metadata graph, and then:

1. recomputes the **daily series** behind every dashboard chart, and checks them against the dashboard,
2. recomputes the **normal / warning / critical status** of each parameter card, and
3. loads the raw data **as linked data** next to the metadata and answers questions with **SPARQL**.

Everything is read-only: the notebook only lists and downloads public files. The metadata and ontology
come from the open-source OBEREK pipelines on GitHub ([NeverBlink-labs/oberek-code](https://github.com/NeverBlink-labs/oberek-code)).

Requirements: `pip install boto3 pandas matplotlib rdflib requests`

In [ ]:
# Settings. The OBEREK metadata and ontology are published with the OBEREK pipelines on GitHub
# (https://github.com/NeverBlink-labs/oberek-code). You can also point these at local copies.
REPO = "https://raw.githubusercontent.com/NeverBlink-labs/oberek-code/main/model"
METADATA = f"{REPO}/oberek-metadata-init/oberek_metadata.ttl"
ONTOLOGY = f"{REPO}/oberek-ontology-init/oberek_ontology.ttl"

STATION = "southwest"   # folder name in the bucket: southwest or north (south sends simulated data)
LOCATION = "SouthWest"  # the same place in the metadata: SouthWest or North
DAYS = 2                # full UTC days to download, plus today so far
HOURS = 6               # hours of linked data to load for the SPARQL questions in section 5

In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone

import boto3
import matplotlib.pyplot as plt
import pandas as pd
import rdflib
import requests
from botocore import UNSIGNED
from botocore.config import Config

ENDPOINT, BUCKET = "https://oberek.neverblink.eu/s3", "oberek-bucket"
s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name="us-east-1",
                  config=Config(signature_version=UNSIGNED, max_pool_connections=8,
                                retries={"max_attempts": 5, "mode": "standard"}))
pd.set_option("display.float_format", lambda x: f"{x:,.2f}" if x == 0 or abs(x) >= 0.01 else f"{x:.1e}")

## 1. Load the metadata graph

The metadata describes every station, sensor and location. The ontology defines the properties they
measure and the indicators built from them. The reference values that the dashboard uses (a
*baseline*, a *warning* and a *critical* boundary per property) are stored there too, as
`nb:PropertyBaseline` resources, each with the source it comes from. So nothing below needs hard-coded numbers.

In [ ]:
g = rdflib.Graph()
g.parse(METADATA, format="turtle")
g.parse(ONTOLOGY, format="turtle")

PREFIXES = """
PREFIX nb:    <https://neverblink.eu/ontologies/oberek/>
PREFIX nbi:   <https://neverblink.eu/oberek/>
PREFIX saref: <https://saref.etsi.org/core/>
PREFIX geo:   <http://www.opengis.net/ont/geosparql#>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
PREFIX s4watr: <https://saref.etsi.org/saref4watr/>
PREFIX bufr4_b13: <http://codes.wmo.int/bufr4/b/13/>
PREFIX owl:   <http://www.w3.org/2002/07/owl#>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

def short(value):
    """URIs as the same short names used in the CSV files (e.g. bufr4_b13:083); literals as Python values."""
    if isinstance(value, rdflib.URIRef):
        return g.namespace_manager.normalizeUri(value)
    return value.toPython() if value is not None else None

def sparql(query, graph=g):
    result = graph.query(PREFIXES + query)
    return pd.DataFrame([[short(v) for v in row] for row in result], columns=[str(v) for v in result.vars])

print(f"{len(g):,} triples loaded")

**Reference values.** One row per property that has a baseline. Warning and critical boundaries are
percentages of deviation from the baseline, counted only in the harmful direction (`nb:Above`,
`nb:Below`, or both when `bidirectional` is true). The last column asks the ontology whether the
property is an input of KEI-1.

In [ ]:
baselines = sparql("""
SELECT ?feature ?name ?unit ?baseline ?warning_pct ?critical_pct ?direction ?bidirectional ?kei1_input WHERE {
  ?b a nb:PropertyBaseline ;
     nb:forProperty ?feature ;
     nb:baselineValue ?baseline ;
     nb:warningThreshold ?warning_pct ;
     nb:criticalThreshold ?critical_pct ;
     nb:bidirectional ?bidirectional .
  OPTIONAL { ?b nb:deviationDirection ?direction }
  OPTIONAL { ?feature rdfs:label ?name FILTER(lang(?name) = "en") }
  OPTIONAL { ?feature qudt:applicableUnit ?unit }
  BIND(EXISTS { nb:KEI-1 nb:hasInputFeature ?feature } AS ?kei1_input)
}
ORDER BY DESC(?kei1_input) ?feature
""").set_index("feature")
for col in ["baseline", "warning_pct", "critical_pct"]:
    baselines[col] = baselines[col].astype(float)
baselines

**The station.** Which sensors belong to the station, what each one measures, and its name in the
three dashboard languages. This is what turns a CSV row like `nbi:Estacion_SouthWest-ec` into
"Sensor de conductividad eléctrica".

In [ ]:
station = sparql(f"""
SELECT ?sensor ?measures ?name_en ?name_es ?name_va WHERE {{
  nbi:Estacion_{LOCATION} saref:consistsOf ?sensor .
  ?sensor saref:observes ?measures ;
          rdfs:label ?name_en, ?name_es, ?name_va .
  FILTER(lang(?name_en) = "en" && lang(?name_es) = "es" && lang(?name_va) = "ca-valencia")
  FILTER NOT EXISTS {{ ?measures a saref:FeatureOfInterest }}
}}
ORDER BY ?sensor ?measures
""")
station

**Plausible ranges.** Each water measurement also has a plausible range (`nb:SensorRange`). OBEREK leaves out any
sensor message with a value outside it, so every reading you download is inside these limits. They are also a handy
first filter if you combine OBEREK data with readings from other sensors.

In [ ]:
sparql("""
SELECT ?feature ?name ?min ?max WHERE {
  ?r a nb:SensorRange ; nb:forProperty ?feature ; nb:plausibleMin ?min ; nb:plausibleMax ?max .
  OPTIONAL { ?feature rdfs:label ?name FILTER(lang(?name) = "en") }
}
ORDER BY ?feature
""")

## 2. Get the raw data

Every message from the station is stored as one file per format under
`raw/iot/<station>/<format>/<YYYY-MM-DD>/`. The CSV version is easiest for pandas.
At SouthWest that's about 1,440 small files a day, so this cell downloads a few thousand files and can
take a minute. The [live-data tutorial](https://neverblink-labs.github.io/oberek-rue/tutorials/live-data/)
explains the folder layout.

In [ ]:
def list_keys(prefix):
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
        keys += [obj["Key"] for obj in page.get("Contents", [])]
    return keys

def read_csv(key):
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))

def read_many(keys):
    with ThreadPoolExecutor(8) as pool:
        frames = list(pool.map(read_csv, keys))
    data = pd.concat(frames, ignore_index=True)
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    return data

today = datetime.now(timezone.utc).date()
days = [today - timedelta(days=n) for n in range(DAYS, -1, -1)]
keys = [k for day in days for k in list_keys(f"raw/iot/{STATION}/csv/{day}/")]
iot = read_many(keys).dropna(subset=["value"])
print(f"{len(keys):,} files, {len(iot):,} readings from {iot.timestamp.min():%Y-%m-%d %H:%M} "
      f"to {iot.timestamp.max():%Y-%m-%d %H:%M} UTC")
iot.head()

**The same data as RDF.** Each CSV file has a Turtle (`.ttl`) twin. It uses the same vocabulary as the
metadata, so you can load one next to the metadata graph and query both together. Here we read the newest
message and join each value with the sensor's Spanish name and the location it belongs to. Neither
of those is in the observation file; both come from the metadata.

In [ ]:
ttl_key = list_keys(f"raw/iot/{STATION}/ttl/{today}/")[-1]  # the newest message
ttl_text = s3.get_object(Bucket=BUCKET, Key=ttl_key)["Body"].read().decode()
combined = g + rdflib.Graph().parse(data=ttl_text, format="turtle")

sparql("""
SELECT ?time ?sensor_es ?property ?value ?unit ?location WHERE {
  ?sensor saref:madeExecution ?obs .
  ?obs saref:hasTimestamp ?time ;
       saref:hasResult [ saref:isValueOfProperty ?property ; saref:hasValue ?value ; saref:isMeasuredIn ?unit ] .
  ?sensor rdfs:label ?sensor_es FILTER(lang(?sensor_es) = "es")
  ?sensor saref:observes ?location . ?location a saref:FeatureOfInterest .
}
ORDER BY ?sensor_es ?property
""", graph=combined)

## 3. Rebuild the dashboard's daily series

Each chart on the dashboard reads a small JSON file with one value per UTC day for the last 30 days:
`<location>/raw/iot/<feature>/latest.json`. OBEREK makes it by averaging every reading of the day. For
water temperature, that includes all four sensors at the station. Let's do the same and compare.

In [ ]:
KEY_SEGMENT = {"SouthWest": "sw", "North": "north", "South": "south"}[LOCATION]

def dashboard_series(feature):
    prefix, name = feature.split(":")
    if name[0].isdigit():         # numeric WMO codes keep their prefix: bufr4_b13-083
        name = f"{prefix}-{name}"
    url = f"{ENDPOINT}/{BUCKET}/{KEY_SEGMENT}/raw/iot/{name}/latest.json"
    points = requests.get(url, timeout=30).json()["data"]
    return pd.Series({pd.Timestamp(p["timestamp"]).date(): p["value"] for p in points}, name="dashboard")

daily = iot.groupby([iot.timestamp.dt.date, "feature"]).value.mean().unstack()
full_days = [d for d in daily.index if d < today]  # today is not finished yet

rows = []
for feature in daily.columns:
    theirs = dashboard_series(feature)
    for day in full_days:
        if day in theirs.index:
            rows.append({"day": day, "feature": feature, "re-derived": daily.loc[day, feature],
                         "dashboard": theirs[day]})
compare = pd.DataFrame(rows)
compare["difference %"] = 100 * (compare["re-derived"] - compare["dashboard"]).abs() / compare["dashboard"].abs()
compare.groupby("feature")["difference %"].max().to_frame("largest difference, % of value")

The differences are rounding noise: OBEREK averages and stores the values as 32-bit floats. Today is left out on purpose: the
dashboard calculates today's point during the day, so it is based on fewer readings than ours. The chart below
shows how much one daily point hides. The minute readings are grey and the daily means are green. The
orange crosses are the dashboard's published points.

In [ ]:
feature = "s4watr:Turbidity"
theirs = dashboard_series(feature)
minute = iot[(iot.feature == feature) & (iot.timestamp.dt.date < today)]  # full days only
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(minute.timestamp, minute.value, color="#9aa59a", lw=0.8, label="every reading")
mid = lambda d: pd.Timestamp(d, tz="UTC") + pd.Timedelta(hours=12)
ax.plot([mid(d) for d in full_days], daily.loc[full_days, feature], "o-", color="#2a7c5e", label="re-derived daily mean")
shown = theirs[[d for d in full_days if d in theirs.index]]
ax.plot([mid(d) for d in shown.index], shown.values, "x", color="#f4a261", ms=10, mew=2, label="dashboard")
ax.set_title(f"{baselines.loc[feature, 'name']} at {LOCATION} (NTU)", loc="left")
ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.3)
plt.show()

## 4. Rebuild the parameter-card colours

Each parameter card on the dashboard is green, orange or red. The rule has three steps:

1. relative deviation from the baseline: `d = (x − baseline) / baseline`,
2. keep only the harmful side: `max(d, 0)` for *Above*, `max(−d, 0)` for *Below*, `|d|` for bidirectional,
3. **warning** if that is ≥ the warning %, **critical** if it is > the critical %.

This is the same rule the KEI pipelines use (see `pipeline/kei-1-processing/manifest.yml` in
[oberek-code](https://github.com/NeverBlink-labs/oberek-code)). The pipelines apply it to the last hour of readings,
every 15 minutes. Here we apply it to the last full day, like the dashboard cards.

In [ ]:
def directed_deviation(value, feature, refs=baselines):
    r = refs.loc[feature]
    d = (value - r.baseline) / (r.baseline + 1e-6)
    if r.bidirectional:
        return abs(d)
    return max(d, 0.0) if r.direction == "nb:Above" else max(-d, 0.0)

def card_status(value, feature, refs=baselines):
    dd, r = directed_deviation(value, feature, refs), refs.loc[feature]
    return "critical" if dd > r.critical_pct / 100 else "warning" if dd >= r.warning_pct / 100 else "normal"

latest_day = full_days[-1]
cards = pd.DataFrame([
    {"parameter": baselines.loc[f, "name"], "latest daily value": daily.loc[latest_day, f],
     "deviation %": 100 * directed_deviation(daily.loc[latest_day, f], f), "status": card_status(daily.loc[latest_day, f], f)}
    for f in daily.columns if f in baselines.index
])
print(f"Status on {latest_day}")
cards

## 5. Ask your own questions with SPARQL

So far we used pandas on the CSV files. The same readings are also published as linked data (TTL), using the
vocabulary of the OBEREK metadata and ontology. Load them into the same graph as the metadata, and every reading is
automatically connected to its sensor, its location, the property it measures, the property's names in three
languages, its reference values, and the indicators it feeds. One SPARQL query can then use all of that at once.

Here we load the last few hours from **both** real stations, SouthWest and North.

In [ ]:
def list_after(prefix, after=""):
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix, StartAfter=after):
        keys += [obj["Key"] for obj in page.get("Contents", [])]
    return keys

def read_text(key):
    return s3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode()

since = datetime.now(timezone.utc) - timedelta(hours=HOURS)
ttl_keys = []
for station in ["southwest", "north"]:
    for day in sorted({since.date(), today}):
        prefix = f"raw/iot/{station}/ttl/{day}/"
        ttl_keys += list_after(prefix, prefix + f"data_{since:%H_%M_%S}.ttl" if day == since.date() else "")

with ThreadPoolExecutor(8) as pool:
    documents = list(pool.map(read_text, ttl_keys))
kg = g + rdflib.Graph()          # a copy of the metadata graph…
for doc in documents:            # …plus every observation file
    kg.parse(data=doc, format="turtle")
print(f"{len(ttl_keys)} files, {len(kg):,} triples in the combined graph")

**What was measured, where, and when?** The observation files only say which sensor made a reading. The location
and the plain name of what it measures come from the metadata.

In [ ]:
sparql("""
SELECT ?place ?measurement (COUNT(?value) AS ?readings) (MIN(?time) AS ?first) (MAX(?time) AS ?last) WHERE {
  ?sensor saref:madeExecution ?obs ;
          saref:observes ?location .
  ?location a saref:FeatureOfInterest ; rdfs:label ?place . FILTER(lang(?place) = "en")
  ?obs saref:hasTimestamp ?time ;
       saref:hasResult [ saref:isValueOfProperty ?property ; saref:hasValue ?value ] .
  ?property rdfs:label ?measurement . FILTER(lang(?measurement) = "en")
}
GROUP BY ?place ?measurement
ORDER BY ?place ?measurement
""", graph=kg)

**How does dissolved oxygen change hour by hour at each station?** SPARQL does the grouping. We only draw the
result.

In [ ]:
hourly = sparql("""
SELECT ?place ?hour (AVG(?value) AS ?oxygen) WHERE {
  ?sensor saref:madeExecution ?obs ;
          saref:observes ?location .
  ?location a saref:FeatureOfInterest ; rdfs:label ?place . FILTER(lang(?place) = "en")
  ?obs saref:hasTimestamp ?time ;
       saref:hasResult [ saref:isValueOfProperty bufr4_b13:083 ; saref:hasValue ?value ] .
  BIND(SUBSTR(STR(?time), 1, 13) AS ?hour)
}
GROUP BY ?place ?hour
ORDER BY ?hour
""", graph=kg)
hourly["hour"] = pd.to_datetime(hourly["hour"] + ":00Z")

fig, ax = plt.subplots(figsize=(9, 3))
for (place, rows), color in zip(hourly.groupby("place"), ["#2a7c5e", "#f4a261"]):
    ax.plot(rows.hour, rows.oxygen.astype(float), "o-", color=color, label=place)
ax.set_title("Dissolved oxygen, hourly average (mg/L)", loc="left")
ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.3)
plt.show()

**How many readings crossed their thresholds?** This query joins every reading with its reference value and
thresholds from the metadata, and follows the ontology to keep only the properties that feed KEI-1
(`nb:KEI-1 nb:hasInputFeature`). The deviation rule is the same one used in section 4.

In [ ]:
sparql("""
SELECT ?place ?measurement (COUNT(?value) AS ?readings)
       (SUM(IF(?deviation >= ?warning / 100, 1, 0)) AS ?over_warning)
       (SUM(IF(?deviation > ?critical / 100, 1, 0)) AS ?over_critical) WHERE {
  ?sensor saref:madeExecution ?obs .
  ?obs saref:hasResult ?result .
  ?result saref:isValueOfProperty ?property ; saref:hasValue ?value .
  nb:KEI-1 nb:hasInputFeature ?property .
  ?b nb:forProperty ?property ;
     nb:baselineValue ?baseline ;
     nb:warningThreshold ?warning ;
     nb:criticalThreshold ?critical ;
     nb:bidirectional ?both .
  OPTIONAL { ?b nb:deviationDirection ?direction }
  ?sensor saref:observes ?location .
  ?location a saref:FeatureOfInterest ; rdfs:label ?place . FILTER(lang(?place) = "en")
  ?property rdfs:label ?measurement . FILTER(lang(?measurement) = "en")
  BIND((?value - ?baseline) / ?baseline AS ?d)
  BIND(IF(?both, ABS(?d), IF(?direction = nb:Above, IF(?d > 0, ?d, 0), IF(?d < 0, -?d, 0))) AS ?deviation)
}
GROUP BY ?place ?measurement
ORDER BY ?place ?measurement
""", graph=kg)

**The same answer in Valencian.** Ask for the labels in another language and the whole result changes language.
The metadata has English (`en`), Spanish (`es`) and Valencian (`ca-valencia`).

In [ ]:
sparql("""
SELECT ?lloc ?mesura (ROUND(AVG(?value) * 100) / 100 AS ?mitjana) WHERE {
  ?sensor saref:madeExecution ?obs ;
          saref:observes ?location .
  ?location a saref:FeatureOfInterest ; rdfs:label ?lloc . FILTER(lang(?lloc) = "ca-valencia")
  ?obs saref:hasResult [ saref:isValueOfProperty ?property ; saref:hasValue ?value ] .
  ?property rdfs:label ?mesura . FILTER(lang(?mesura) = "ca-valencia")
}
GROUP BY ?lloc ?mesura
ORDER BY ?mesura ?lloc
""", graph=kg)

**Save your work.** Write the re-derived daily series and the hourly averages to CSV:

In [ ]:
daily.to_csv(f"oberek_{STATION}_daily.csv")
hourly.to_csv("oberek_oxygen_hourly.csv", index=False)
print("saved", f"oberek_{STATION}_daily.csv", "oberek_oxygen_hourly.csv")

## Good to know

* **The data is provisional.** It is published automatically, before quality checks, and the
  baselines are under scientific review
  ([data use notice](https://oberek.neverblink.eu/help#data-use-notice)).
* **The indicators (KEI-1, KEI-2, KEI-3)** combine these statuses into levels from 1 to 5. The dashboard's
  [Help page](https://oberek.neverblink.eu/help), section "How the KEIs Are Calculated", explains the method.
  OBEREK publishes its own results every 15 minutes under `keis/kei-1/csv/`, `keis/kei-2/csv/` and `keis/kei-3/csv/`.
  The South water station currently sends simulated data, so the results for the South and Global locations are not real yet.
* **The pipelines are open source.** The exact queries behind the indicators are in
  [NeverBlink-labs/oberek-code](https://github.com/NeverBlink-labs/oberek-code): the KEIs in `pipeline/kei-*-processing/`,
  the rice blast index in `pipeline/pri/`, and the ingestion of each source in `pipeline/*-ingestion/`. Start there if you
  want to change a rule, try other reference values, or set up the same indicators for another site.
* **Need a longer period?** A snapshot of everything collected up to 15 October 2026 is on Zenodo,
  in the same folder layout and formats: [doi.org/10.5281/zenodo.23418104](https://doi.org/10.5281/zenodo.23418104).
  Its README lists the period covered for each source. Use it for anything before mid-October 2026.
* **Weather, satellite and flow data** are in the same bucket and are loaded the same way. The
  [live-data tutorial](https://neverblink-labs.github.io/oberek-rue/tutorials/live-data/) lists the folders.